In [13]:
import ast
import json
from pathlib import Path
from types import SimpleNamespace
from os import listdir
import matplotlib.pyplot as plt
import numpy as np
from physics_definitions import load_physics_definitions, visible_limits

In [14]:
ROOT = Path.cwd().parent

In [15]:

PHYSICS_STAGE = ROOT / "stages" / "s01_physics.py"
OUTPUT_DIR = ROOT / "plot_potentials" / "graphs_ID"
SELECTED_NAMES = {
    "ANALYTICAL_FAMILIES",
    "FUNCTIONAL_FAMILIES",
    "TRAIN_FAMILIES",
    "REPARAMETERIZED_SEEN_FAMILIES",
    "UNSEEN_ANALYTICAL_FAMILIES",
    "HELD_OUT_FAMILIES",
    "PotentialSpec",
    "_coefficients",
    "make_spec",
    "potential_value",
    "geometry_grid",
    "sample_potential_safely",
}

In [16]:
definitions = load_physics_definitions(PHYSICS_STAGE, SELECTED_NAMES)
    
families = definitions["TRAIN_FAMILIES"]
make_spec = definitions["make_spec"]
geometry_grid = definitions["geometry_grid"]
sample_potential_safely = definitions["sample_potential_safely"]

In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [18]:
def visible_limits(values: np.ndarray) -> tuple[float, float]:
    lower, upper = np.quantile(values, [0.005, 0.995])
    if not np.isfinite(lower) or not np.isfinite(upper) or lower == upper:
        lower, upper = float(np.min(values)), float(np.max(values))
    padding = max((upper - lower) * 0.08, 0.1)
    return float(lower - padding), float(upper + padding)

In [25]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 16,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

In [26]:
rng = np.random.default_rng(20260908)
manifest = []
for index, family in enumerate(families, start=1):
    spec = make_spec(family, rng.random(16), base_id=f"plot-{index:02d}")
    x = geometry_grid(spec, n_nodes=2_001)
    values, valid, singular = sample_potential_safely(spec, x)
    plotted_x, plotted_values = x[valid], values[valid]

    figure, axis = plt.subplots(figsize=(10, 5.5), layout="constrained")
    axis.plot(plotted_x, plotted_values, color="#006d77", linewidth=1.8)
    axis.axhline(0.0, color="#6b7280", linewidth=0.8, zorder=0)
    if singular.any():
        axis.scatter(x[singular], values[singular], color="#d1495b", marker="x", s=45,
                     label="singular boundary (excluded)", zorder=3)
        axis.legend(frameon=False, loc="best")
    axis.set(title=family.replace("_", " ").title(), xlabel="x", ylabel="V(x)",
             ylim=visible_limits(plotted_values))
    axis.grid(alpha=0.22)
    parameters = ", ".join(f"{key}={value}" for key, value in spec.parameters.items())
    axis.text(0.01, 0.01, parameters, transform=axis.transAxes, fontsize=8,
              va="bottom", ha="left", wrap=True)
    filename = f"{index:02d}_{family}.png"
    figure.savefig(OUTPUT_DIR / filename, dpi=180)
    plt.close(figure)
    manifest.append({
        "file": filename,
        "family": family,
        "parameters": spec.parameters,
        "geometry": spec.geometry,
        "domain": [spec.domain_left, spec.domain_right],
        "singular_boundaries": int(singular.sum()),
    })

In [24]:
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Saved {len(manifest)} potential plots to {OUTPUT_DIR}")

Saved 27 potential plots to /home/clouduser/psi_Jepa_V08/plot_potentials/graphs_ID
